# M4.Ex2: Penguins Classification (sklearn — PyCaret-style workflow)

> **Note:** PyCaret requires Python 3.9–3.11 and does **not** support Python 3.12+.
> This notebook replicates the exact same PyCaret classification workflow using **scikit-learn**.

## Exercise

Three experiments on the **Palmer Penguins Dataset**:

1. X = Flipper Length & Bill Length → y = Species
2. X = Body Mass & Species → y = Sex
3. X = island, bill_length_mm, bill_depth_mm, flipper_length_mm, body_mass_g → y = Species & Sex (multi-label)

Workflow per experiment: **Setup → Compare Models → Analyze → Predict → Save**

## Palmer Penguins Dataset
- Features: 4 numerical, 2 categorical
- Target: `species` (3 classes) or `sex` (2 classes)
- Size: 344 samples

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

### Load the Data

In [ ]:
penguins = sns.load_dataset('penguins')
print(f'Shape: {penguins.shape}')
print(f'Missing values:\n{penguins.isnull().sum()}')
display(penguins.head())

### Helper Functions (reused across all 3 experiments)

In [ ]:
CLASSIFIERS = {
    'Logistic Regression' : LogisticRegression(max_iter=500, random_state=42),
    'Decision Tree'       : DecisionTreeClassifier(random_state=42),
    'Random Forest'       : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Extra Trees'         : ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting'   : GradientBoostingClassifier(random_state=42),
    'KNN'                 : KNeighborsClassifier(),
    'SVM'                 : SVC(probability=True, random_state=42),
    'Naive Bayes'         : GaussianNB(),
}

def make_preprocessor(X):
    """Build a ColumnTransformer that handles numerical and categorical columns."""
    num_cols = X.select_dtypes(include=['number']).columns.tolist()
    cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
    transformers = []
    if num_cols:
        transformers.append(('num', Pipeline([
            ('imp', SimpleImputer(strategy='mean')),
            ('sc',  StandardScaler())
        ]), num_cols))
    if cat_cols:
        transformers.append(('cat', Pipeline([
            ('imp', SimpleImputer(strategy='most_frequent')),
            ('enc', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_cols))
    return ColumnTransformer(transformers=transformers)

def compare_models(X_train, y_train, preprocessor):
    """5-fold CV benchmark of all classifiers. Returns leaderboard + fitted pipelines."""
    results, pipes = [], {}
    for name, clf in CLASSIFIERS.items():
        pipe = Pipeline([('pre', preprocessor), ('clf', clf)])
        cv   = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)
        pipes[name] = pipe
        results.append({'Model': name, 'CV Accuracy Mean': round(cv.mean(), 4), 'CV Accuracy Std': round(cv.std(), 4)})
    board = pd.DataFrame(results).sort_values('CV Accuracy Mean', ascending=False).reset_index(drop=True)
    return board, pipes

def analyze_model(pipe, X_test, y_test, title=''):
    """Print classification report and plot confusion matrix."""
    y_pred = pipe.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f'=== {title} — Test Accuracy: {acc:.4f} ({acc*100:.1f}%) ===')
    print(classification_report(y_test, y_pred))
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, colorbar=False,
                                            cmap='Blues')
    ax.set_title(f'Confusion Matrix — {title}')
    plt.tight_layout()
    plt.show()
    return acc

print('Helper functions ready.')

---
## Experiment 1: Flipper Length & Bill Length → Species

In [ ]:
# --- Setup ---
df1 = penguins[['flipper_length_mm', 'bill_length_mm', 'species']].dropna()

X1 = df1[['flipper_length_mm', 'bill_length_mm']]
y1 = df1['species']

X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y1, test_size=0.2, random_state=42, stratify=y1
)

pre1 = make_preprocessor(X1)

print(f'Samples: {len(df1)} | Train: {len(X1_train)} | Test: {len(X1_test)}')
print(f'Classes: {y1.unique().tolist()}')

In [ ]:
# --- Compare Models ---
print('Experiment 1 — Benchmarking classifiers...\n')
board1, pipes1 = compare_models(X1_train, y1_train, pre1)
display(board1)

In [ ]:
# --- Analyze Best Model ---
best_name1 = board1.iloc[0]['Model']
best_pipe1 = pipes1[best_name1]
best_pipe1.fit(X1_train, y1_train)

analyze_model(best_pipe1, X1_test, y1_test, title=f'Exp1: {best_name1}')

In [ ]:
# --- Predictions on new data ---
new1 = pd.DataFrame([{'flipper_length_mm': 200.0, 'bill_length_mm': 45.0}])
print(f'New penguin prediction: {best_pipe1.predict(new1)[0]}')

In [ ]:
# --- Save Model ---
joblib.dump(best_pipe1, 'exp1_species_model.pkl')
print('Experiment 1 model saved: exp1_species_model.pkl')

---
## Experiment 2: Body Mass & Species → Sex

In [ ]:
# --- Setup ---
df2 = penguins[['body_mass_g', 'species', 'sex']].dropna()

X2 = df2[['body_mass_g', 'species']]   # 1 numerical + 1 categorical
y2 = df2['sex']

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)

pre2 = make_preprocessor(X2)  # handles both num + cat columns

print(f'Samples: {len(df2)} | Train: {len(X2_train)} | Test: {len(X2_test)}')
print(f'Classes: {y2.unique().tolist()}')
display(X2.head())

In [ ]:
# --- Compare Models ---
print('Experiment 2 — Benchmarking classifiers...\n')
board2, pipes2 = compare_models(X2_train, y2_train, pre2)
display(board2)

In [ ]:
# --- Analyze Best Model ---
best_name2 = board2.iloc[0]['Model']
best_pipe2 = pipes2[best_name2]
best_pipe2.fit(X2_train, y2_train)

analyze_model(best_pipe2, X2_test, y2_test, title=f'Exp2: {best_name2}')

In [ ]:
# --- Predictions on new data ---
new2 = pd.DataFrame([{'body_mass_g': 4200.0, 'species': 'Adelie'}])
print(f'Predicted sex: {best_pipe2.predict(new2)[0]}')

In [ ]:
# --- Save Model ---
joblib.dump(best_pipe2, 'exp2_sex_model.pkl')
print('Experiment 2 model saved: exp2_sex_model.pkl')

---
## Experiment 3: island + measurements → Species & Sex (Multi-label Classification)

In [ ]:
# --- Setup ---
# Multi-label: predict BOTH species and sex simultaneously
from sklearn.multioutput import MultiOutputClassifier

feature_cols = ['island', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
target_cols  = ['species', 'sex']

df3 = penguins[feature_cols + target_cols].dropna()

X3 = df3[feature_cols]
y3 = df3[target_cols]   # 2-column target DataFrame

X3_train, X3_test, y3_train, y3_test = train_test_split(
    X3, y3, test_size=0.2, random_state=42
)

pre3 = make_preprocessor(X3)

print(f'Samples: {len(df3)} | Train: {len(X3_train)} | Test: {len(X3_test)}')
print(f'Features: {feature_cols}')
print(f'Targets : {target_cols}')

In [ ]:
# --- Compare Models (Multi-label) ---
# MultiOutputClassifier wraps any single-output classifier to predict multiple targets
multi_candidates = {
    'Random Forest'     : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Extra Trees'       : ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting' : GradientBoostingClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42),
    'Decision Tree'     : DecisionTreeClassifier(random_state=42),
    'KNN'               : KNeighborsClassifier(),
}

results3, pipes3 = [], {}
print('Experiment 3 — Multi-label benchmark...\n')
for name, clf in multi_candidates.items():
    multi_clf = MultiOutputClassifier(clf, n_jobs=-1)
    pipe = Pipeline([('pre', pre3), ('clf', multi_clf)])
    # Accuracy here = exact match (both species AND sex correct)
    cv = cross_val_score(pipe, X3_train, y3_train, cv=5, scoring='accuracy', n_jobs=-1)
    pipes3[name] = pipe
    results3.append({'Model': name, 'CV Exact-Match Acc': round(cv.mean(), 4), 'Std': round(cv.std(), 4)})
    print(f'{name:<22} Exact-Match Acc = {cv.mean():.4f} +/- {cv.std():.4f}')

board3 = pd.DataFrame(results3).sort_values('CV Exact-Match Acc', ascending=False).reset_index(drop=True)
print('\n=== Leaderboard ===')
display(board3)

In [ ]:
# --- Analyze Best Multi-label Model ---
best_name3 = board3.iloc[0]['Model']
best_pipe3 = pipes3[best_name3]
best_pipe3.fit(X3_train, y3_train)

y3_pred = best_pipe3.predict(X3_test)
y3_pred_df = pd.DataFrame(y3_pred, columns=target_cols)

print(f'=== Exp3: {best_name3} — Multi-label Results ===')
for col in target_cols:
    acc = accuracy_score(y3_test[col].values, y3_pred_df[col].values)
    print(f'\n  Target: {col} — Accuracy: {acc:.4f}')
    print(classification_report(y3_test[col].values, y3_pred_df[col].values))

exact_match = accuracy_score(y3_test.values, y3_pred)
print(f'Exact-match accuracy (both correct): {exact_match:.4f}')

In [ ]:
# Confusion matrices for each target
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, col in zip(axes, target_cols):
    ConfusionMatrixDisplay.from_predictions(
        y3_test[col].values, y3_pred_df[col].values,
        ax=ax, colorbar=False, cmap='Blues'
    )
    ax.set_title(f'Confusion Matrix — {col}')
plt.suptitle(f'Exp3: {best_name3}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- Predict on a new penguin ---
new3 = pd.DataFrame([{
    'island'           : 'Biscoe',
    'bill_length_mm'   : 46.0,
    'bill_depth_mm'    : 14.0,
    'flipper_length_mm': 210.0,
    'body_mass_g'      : 4200.0
}])

pred3 = best_pipe3.predict(new3)
print(f'Predicted species: {pred3[0][0]}')
print(f'Predicted sex    : {pred3[0][1]}')

In [ ]:
# --- Save Model ---
joblib.dump(best_pipe3, 'exp3_multilabel_model.pkl')
print('Experiment 3 model saved: exp3_multilabel_model.pkl')

# Verify reload
loaded3 = joblib.load('exp3_multilabel_model.pkl')
v = loaded3.predict(new3)
print(f'Loaded model — Species: {v[0][0]}, Sex: {v[0][1]}')

---
## Summary of All 3 Experiments

In [ ]:
summary = pd.DataFrame([
    {'Experiment': 'Exp 1', 'Features': 'flipper_length + bill_length',  'Target': 'species', 'Best Model': best_name1},
    {'Experiment': 'Exp 2', 'Features': 'body_mass + species',           'Target': 'sex',     'Best Model': best_name2},
    {'Experiment': 'Exp 3', 'Features': 'island + 4 measurements',       'Target': 'species + sex (multi-label)', 'Best Model': best_name3},
])
display(summary)